In [6]:
import os
from datasets import load_dataset
from transformers import AutoTokenizer

repo_id = "MrBigBrane/LongBench-v2-32k-CoT"
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

# Load a generic background corpus to act as the "haystack" padding
print("Downloading background filler corpus (WikiText)...")
wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
filler_text = "\n".join(wiki["text"])
# Tokenizing the entire wikitext corpus gives us a massive pool of ~2M background tokens
global_filler_tokens = tokenizer.encode(filler_text, add_special_tokens=False)

def process_and_pad_example(example, tok, filler_pool, max_tokens=32700):
    import random # <-- Imported locally for Windows multiprocessing workers
    
    question = example.get("question", "")
    choices = (
        f"A) {example.get('choice_A', '')}\n"
        f"B) {example.get('choice_B', '')}\n"
        f"C) {example.get('choice_C', '')}\n"
        f"D) {example.get('choice_D', '')}"
    )
    target = example.get("answer", "")
    
    header = "Read the following text and answer the multiple-choice question. Let's think step-by-step.\n\nBackground Context:\n"
    footer = f"\n\nQuestion: {question}\n{choices}\n\nAnswer:"
    
    header_tokens = tok.encode(header, add_special_tokens=False)
    footer_tokens = tok.encode(footer, add_special_tokens=False)
    
    raw_context = example.get("context", "")
    context_tokens = tok.encode(raw_context, add_special_tokens=False)
    
    # Calculate how much generic padding is needed to hit exactly 32,700
    current_length = len(header_tokens) + len(context_tokens) + len(footer_tokens)
    padding_needed = max_tokens - current_length
    
    if padding_needed > 0:
        # Grab a random contiguous chunk of background tokens to act as padding
        start_idx = random.randint(0, len(filler_pool) - padding_needed - 1)
        padding_tokens = filler_pool[start_idx : start_idx + padding_needed]
        
        # Prepend the padding so the actual context sits closer to the question
        final_tokens = header_tokens + padding_tokens + context_tokens + footer_tokens
    else:
        # If it's naturally over 32,700, cap it safely
        final_tokens = (header_tokens + context_tokens + footer_tokens)[:max_tokens]
        
    final_prompt = tok.decode(final_tokens)
    
    return {
        "input": final_prompt,
        "outputs": [target],
        "padded_tokens_added": padding_needed if padding_needed > 0 else 0
    }

def filter_short(example, tok):
    # Only keep examples under 30,000 tokens so we can pad them UP
    return len(tok.encode(example["context"], add_special_tokens=False)) < 30000

if __name__ == "__main__":
    num_cpus = max(1, os.cpu_count() - 2)
    print(f"Downloading LongBench-v2...")
    dataset = load_dataset("THUDM/LongBench-v2", split="train")

    print("Filtering dataset for examples under 30k tokens...")
    dataset = dataset.filter(
        filter_short,
        fn_kwargs={"tok": tokenizer},
        num_proc=num_cpus
    )

    print(f"Padding {len(dataset)} examples to exactly 32,700 tokens...")
    formatted = dataset.map(
        process_and_pad_example,
        fn_kwargs={
            "tok": tokenizer, 
            "filler_pool": global_filler_tokens,
            "max_tokens": 32700
        },
        remove_columns=dataset.column_names,
        num_proc=num_cpus
    )

    formatted.push_to_hub(repo_id, split="train")
    print(f"Successfully uploaded {len(formatted)} padded examples to {repo_id}!")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2517233 > 131072). Running this sequence through the model will result in indexing errors


Filtering dataset for examples under 30k tokens...
Padding 106 examples to exactly 32,700 tokens...


Map (num_proc=22): 100%|██████████| 106/106 [01:35<00:00,  1.10 examples/s]
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 11.82ba/s]
Processing Files (1 / 1): 100%|██████████| 7.33MB / 7.33MB,  559kB/s  
New Data Upload: 100%|██████████| 7.33MB / 7.33MB,  559kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:04<00:00,  4.25s/ shards]


Successfully uploaded 106 padded examples to MrBigBrane/LongBench-v2-32k-CoT!


In [8]:
from datasets import load_dataset

# Force a fresh download to overwrite the cached schema
test_dataset = load_dataset(
    "MrBigBrane/LongBench-v2-32k-CoT", 
    split="train",
    download_mode="force_redownload"
)

print(f"Total examples ready for sweep: {len(test_dataset)}")
print(f"Sample prompt length (characters): {len(test_dataset[0]['input'])}")
print(f"Padded tokens added to first example: {test_dataset[0]['padded_tokens_added']}")

Generating train split:   0%|          | 0/106 [00:00<?, ? examples/s]Failed to read file 'C:\Users\theep\.cache\huggingface\hub\datasets--MrBigBrane--LongBench-v2-32k-CoT\snapshots\ace22e6650eb7c499e29d2ced45a93adec6a7061\data\train-00000-of-00001.parquet' with error CastError: Couldn't cast
input: string
outputs: list<element: string>
  child 0, element: string
padded_tokens_added: int64
-- schema metadata --
huggingface: '{"info": {"features": {"input": {"dtype": "string", "_type"' + 154
to
{'input': Value('string'), 'outputs': List(Value('string')), 'original_token_length': Value('int64')}
because column names don't match
Generating train split:   0%|          | 0/106 [00:00<?, ? examples/s]


DatasetGenerationError: An error occurred while generating the dataset

In [10]:
from datasets import load_dataset

# Load the raw Parquet file directly, ignoring any cached repository schema
test_dataset = load_dataset(
    "parquet", 
    data_files="hf://datasets/MrBigBrane/LongBench-v2-32k-CoT/data/train-*", 
    split="train"
)

print(f"Total examples ready for sweep: {len(test_dataset)}")
print(f"Sample prompt length (characters): {len(test_dataset[0]['input'])}")
print(f"Padded tokens added to first example: {test_dataset[0]['padded_tokens_added']}")

Generating train split: 106 examples [00:00, 4704.17 examples/s]

Total examples ready for sweep: 106
Sample prompt length (characters): 135072
Padded tokens added to first example: 8535
